# TSL-51 Inference Demo

Real-time inference demonstration for Thai Sign Language recognition.

## 1. Setup (Run 01_setup.ipynb first)

In [ ]:
import os
import sys
import torch
import numpy as np
import cv2
import mediapipe as mp
from google.colab.patches import cv2_imshow

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/TSL'
os.chdir(WORK_DIR)
sys.path.insert(0, os.path.join(WORK_DIR, 'src'))

print(f'Working directory: {os.getcwd()}')

## 2. Load Trained Model

In [ ]:
# Specify model path (update with your trained model)
MODEL_PATH = '/content/drive/MyDrive/TSL/models/tsl51_gru_20260519_214700.pt'

checkpoint = torch.load(MODEL_PATH, map_location='cpu')
config = checkpoint['config']
classes = checkpoint['classes']
mean = checkpoint.get('mean', 0)
std = checkpoint.get('std', 1)

print(f'Model loaded from: {MODEL_PATH}')
print(f'Number of classes: {len(classes)}')
print(f'Classes: {classes}')

## 3. Rebuild Model

In [ ]:
from src.core.models import GRUModel, MLPModel

input_dim = 162  # Default feature dimension
num_classes = len(classes)

if config['model_type'] == 'gru':
    model = GRUModel(
        input_dim=input_dim,
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        num_classes=num_classes,
        dropout=config['dropout']
    )
elif config['model_type'] == 'mlp':
    model = MLPModel(
        input_dim=input_dim,
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_layers'],
        num_classes=num_classes,
        dropout=config['dropout']
    )

model.load_state_dict(checkpoint['state_dict'])
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f'Model rebuilt and loaded on {device}')

## 4. Initialize MediaPipe

In [ ]:
# Initialize MediaPipe Hands and Pose
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print('MediaPipe initialized')

## 5. Feature Extraction Function

In [ ]:
def extract_features(frame):
    """Extract 162-dimensional features from frame."""
    # Convert BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Get hand landmarks
    hand_results = hands.process(rgb_frame)
    
    # Get pose landmarks
    pose_results = pose.process(rgb_frame)
    
    features = np.zeros(162)
    
    # Extract left hand (63 features: 21 landmarks * 3 coords)
    if hand_results.multi_hand_landmarks:
        for idx, hand_landmarks in enumerate(hand_results.multi_hand_landmarks):
            if idx >= 2:  # Max 2 hands
                break
            
            # Determine if left or right hand
            # (simplified - in practice use handedness)
            hand_idx = 0 if idx == 0 else 1
            
            for i, landmark in enumerate(hand_landmarks.landmark):
                if i < 21:
                    base_idx = hand_idx * 63 + i * 3
                    features[base_idx] = landmark.x
                    features[base_idx + 1] = landmark.y
                    features[base_idx + 2] = landmark.z
    
    # Extract pose (36 features: 12 landmarks * 3 coords)
    if pose_results.pose_landmarks:
        pose_landmarks = [
            11, 12, 13, 14, 15, 16,  # shoulders, elbows, wrists
            0, 1, 2, 3, 4, 5          # face landmarks (brows, mouth)
        ]
        
        for i, idx in enumerate(pose_landmarks):
            if idx < len(pose_results.pose_landmarks.landmark):
                landmark = pose_results.pose_landmarks.landmark[idx]
                base_idx = 126 + i * 3
                features[base_idx] = landmark.x
                features[base_idx + 1] = landmark.y
                features[base_idx + 2] = landmark.z
    
    return features

## 6. Prediction Function

In [ ]:
def predict_sign(features):
    """Predict Thai sign from features."""
    # Normalize
    features_normalized = (features - mean) / std
    
    # Convert to tensor
    features_tensor = torch.FloatTensor(features_normalized).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(features_tensor)
        probs = outputs.softmax(dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs[0][pred_idx].item()
    
    return classes[pred_idx], confidence

## 7. Upload Video for Prediction

In [ ]:
from google.colab import files

print('Upload a video file for prediction...')
uploaded = files.upload()

if uploaded:
    video_path = list(uploaded.keys())[0]
    print(f'Video uploaded: {video_path}')

## 8. Process Video

In [ ]:
# Open video
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f'Video FPS: {fps}')
print(f'Total frames: {frame_count}')

# Process frames
predictions = []
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Extract features
    features = extract_features(frame)
    
    # Predict
    sign, confidence = predict_sign(features)
    predictions.append({
        'frame': frame_idx,
        'sign': sign,
        'confidence': confidence
    })
    
    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f'Processed {frame_idx}/{frame_count} frames')

cap.release()
print(f'\nProcessed {len(predictions)} frames')

## 9. Analyze Predictions

In [ ]:
from collections import Counter

# Get most common prediction
signs = [p['sign'] for p in predictions]
sign_counts = Counter(signs)

print('\nPrediction Statistics:')
print(f'Total frames: {len(predictions)}')
print(f'Unique signs detected: {len(sign_counts)}')

print('\nTop predictions:')
for sign, count in sign_counts.most_common(10):
    percentage = (count / len(predictions)) * 100
    print(f'{sign}: {count} frames ({percentage:.1f}%)')

# Final prediction (most common)
final_prediction = sign_counts.most_common(1)[0][0]
print(f'\nFinal prediction: {final_prediction}')

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot prediction timeline
frame_indices = [p['frame'] for p in predictions]
sign_labels = [p['sign'] for p in predictions]

plt.figure(figsize=(15, 6))
plt.plot(frame_indices, [hash(s) % 100 for s in sign_labels], 'o-', alpha=0.5)
plt.xlabel('Frame')
plt.ylabel('Sign Hash')
plt.title('Sign Prediction Timeline')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot confidence scores
confidences = [p['confidence'] for p in predictions]
plt.figure(figsize=(15, 6))
plt.plot(frame_indices, confidences)
plt.xlabel('Frame')
plt.ylabel('Confidence')
plt.title('Prediction Confidence Timeline')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
import json
from datetime import datetime

results = {
    'video_path': video_path,
    'model_path': MODEL_PATH,
    'total_frames': len(predictions),
    'final_prediction': final_prediction,
    'sign_distribution': dict(sign_counts),
    'predictions': predictions,
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S')
}

results_path = os.path.join(WORK_DIR, 'results', f'inference_{results["timestamp"]}.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Inference results saved to: {results_path}')

## 12. Cleanup

In [ ]:
hands.close()
pose.close()
print('MediaPipe resources released')

## Inference Complete